In [ ]:
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from tqdm.auto import tqdm
tqdm.pandas()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Datasets/Tripadvisor_hotel_reviews.csv')
df

,Review,Rating
0,nice hotel expensive parking got good deal sta...,4
1,ok nothing special charge diamond member hilto...,2
2,nice rooms not 4* experience hotel monaco seat...,3
3,"unique, great stay, wonderful time hotel monac...",5
4,"great stay great stay, went seahawk game aweso...",5
...,...,...
20486,"best kept secret 3rd time staying charm, not 5...",5
20487,great location price view hotel great quick pl...,4
20488,"ok just looks nice modern outside, desk staff ...",2
20489,hotel theft ruined vacation hotel opened sept ...,1


In [ ]:
df['Label'] = df['Rating'] - 1
df

,Review,Rating,Label
0,nice hotel expensive parking got good deal sta...,4,3
1,ok nothing special charge diamond member hilto...,2,1
2,nice rooms not 4* experience hotel monaco seat...,3,2
3,"unique, great stay, wonderful time hotel monac...",5,4
4,"great stay great stay, went seahawk game aweso...",5,4
...,...,...,...
20486,"best kept secret 3rd time staying charm, not 5...",5,4
20487,great location price view hotel great quick pl...,4,3
20488,"ok just looks nice modern outside, desk staff ...",2,1
20489,hotel theft ruined vacation hotel opened sept ...,1,0


In [ ]:
df.drop(columns=['Rating'], inplace=True, axis=1)
df

,Review,Label
0,nice hotel expensive parking got good deal sta...,3
1,ok nothing special charge diamond member hilto...,1
2,nice rooms not 4* experience hotel monaco seat...,2
3,"unique, great stay, wonderful time hotel monac...",4
4,"great stay great stay, went seahawk game aweso...",4
...,...,...
20486,"best kept secret 3rd time staying charm, not 5...",4
20487,great location price view hotel great quick pl...,3
20488,"ok just looks nice modern outside, desk staff ...",1
20489,hotel theft ruined vacation hotel opened sept ...,0


In [ ]:
df.describe()

,Label
count,20491.000000
mean,2.952223
std,1.233030
min,0.000000
25%,2.000000
50%,3.000000
75%,4.000000
max,4.000000


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20491 entries, 0 to 20490
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Review  20491 non-null  object
 1   Label   20491 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 320.3+ KB


In [ ]:
df['Label'].value_counts()

,count
Label,
4,9054
3,6039
2,2184
1,1793
0,1421


In [ ]:
train_text, validation_text, train_labels, validation_labels = train_test_split(df['Review'], df['Label'], test_size=0.2, random_state=42)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
train_encoding = tokenizer(list(train_text), truncation=True, padding=True, max_length=256)

In [ ]:
validation_encoding = tokenizer(list(validation_text), truncation=True, padding=True, max_length=256)

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

In [ ]:
class ReviewDataset(Dataset):
  def __init__(self, encodings, labels):
    self.encodings = encodings
    self.labels = labels
  def __getitem__(self, idx):
    item = {key: torch.tensor(validation[idx]) for key, validation in self.encodings.items()}
    item['labels'] = torch.tensor(self.labels[idx])
    return item
  def __len__(self):
    return len(self.labels)

In [ ]:
train_dataset = ReviewDataset(train_encoding, list(train_labels))
validation_dataset = ReviewDataset(validation_encoding, list(validation_labels))

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(105879, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

In [ ]:
total_steps = len(train_dataset)*3   #it is epoch

In [ ]:
from transformers import get_linear_schedule_with_warmup

In [ ]:
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
validation_dataset = DataLoader(validation_dataset, batch_size=16)

In [ ]:
for epoch in range(3):
  model.train()
  total_loss=0
  for batch in tqdm(train_loader):
    optimizer.zero_grad()
    batch = {k:v.to(device) for k,v in batch.items()}
    outputs = model(**batch)
    loss = outputs[0] # Correct: outputs[0] is loss during training
    total_loss += loss.item()
    loss.backward()
    optimizer.step()
    scheduler.step()
  print(f'Epoch: {epoch}, Loss: {total_loss/len(train_loader)}')
  model.eval()
  predictions = []
  true_labels = []
  with torch.no_grad():
    for batch in tqdm(validation_dataset, desc='Validation'):
      batch = {k:v.to(device) for k,v in batch.items()}
      outputs = model(**batch)
      logits = outputs[1]
      prediction = torch.argmax(logits, dim=1).cpu().numpy()
      predictions.extend(prediction)
      true_labels.extend(batch['labels'].cpu().numpy())
  Accuracy = accuracy_score(true_labels, predictions)
  print(f'Accuracy: {Accuracy:.4f}')

print(classification_report(true_labels, predictions))

  0%|          | 0/1025 [00:00<?, ?it/s]

Epoch: 0, Loss: 0.7592147314839247


Validation:   0%|          | 0/257 [00:00<?, ?it/s]

Accuracy: 0.6633


  0%|          | 0/1025 [00:00<?, ?it/s]

...

Accuracy: 0.6663


  0%|          | 0/1025 [00:00<?, ?it/s]

Epoch: 2, Loss: 0.5703275042336161


Validation:   0%|          | 0/257 [00:00<?, ?it/s]

Accuracy: 0.6611
              precision    recall  f1-score   support

           0       0.76      0.62      0.68       292
           1       0.48      0.48      0.48       333
           2       0.44      0.42      0.43       432
           3       0.59      0.56      0.58      1252
           4       0.77      0.83      0.80      1790

    accuracy                           0.66      4099
   macro avg       0.61      0.58      0.59      4099
weighted avg       0.66      0.66      0.66      4099



To improve result we need more epochs.